# Variational Autoencoders — Lab Notebook

This lab builds a VAE from scratch on MNIST and explores the key properties
that distinguish VAEs from plain autoencoders.

**What you will implement and explore:**
1. A plain Autoencoder (AE) — baseline
2. A Variational Autoencoder (VAE) — with reparameterisation trick and ELBO loss
3. Latent space visualisation — AE vs VAE
4. Generation — sampling from the prior
5. Latent space interpolation
6. The latent grid — decoding a 2D map of the digit manifold
7. β-VAE — trading reconstruction quality for latent structure

**Runtime:** use GPU (Runtime → Change runtime type → T4 GPU). Training takes ~2 min per model.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Data — MNIST

In [ ]:
transform = transforms.ToTensor()   # pixels in [0, 1]

train_data = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'Training samples : {len(train_data)}')
print(f'Test samples     : {len(test_data)}')
print(f'Image shape      : {train_data[0][0].shape}  (C x H x W)')

## 2. Plain Autoencoder (baseline)

A plain AE compresses $\mathbf{x}$ into a fixed vector $\mathbf{z} = f(\mathbf{x})$
and reconstructs $\hat{\mathbf{x}} = g(\mathbf{z})$.
There is no probabilistic structure — $\mathbf{z}$ is a deterministic point.

**Loss:** reconstruction only — $\mathcal{L} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 400), nn.ReLU(),
            nn.Linear(400, latent_dim),
        )
    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 400), nn.ReLU(),
            nn.Linear(400, 784), nn.Sigmoid(),  # output in [0, 1]
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class PlainAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z    = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z


def train_ae(model, loader, epochs=10, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []
    for ep in range(1, epochs + 1):
        total = 0
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad()
            xhat, _ = model(x)
            loss = F.mse_loss(xhat, x)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
        avg = total / len(loader.dataset)
        losses.append(avg)
        print(f'  Epoch {ep:2d}  loss = {avg:.5f}')
    return losses


LATENT_DIM = 2   # 2D so we can visualise

ae = PlainAE(LATENT_DIM).to(DEVICE)
print('Training plain AE ...')
ae_losses = train_ae(ae, train_loader, epochs=10)

## 3. Variational Autoencoder

The VAE adds two key ideas:

**Probabilistic encoder:** instead of a single point $\mathbf{z}$, the encoder
outputs the parameters of a Gaussian:
$$q_\phi(\mathbf{z}|\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}(\mathbf{x}),\,\operatorname{diag}(\boldsymbol{\sigma}^2(\mathbf{x})))$$

**Reparameterisation trick:** to allow gradients to flow through the sampling step,
we write:
$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$
The randomness is in $\boldsymbol{\varepsilon}$, which does not depend on any parameters,
so gradients flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ normally.

**ELBO loss:**
$$\mathcal{L} = \underbrace{\mathbb{E}[\log p(\mathbf{x}|\mathbf{z})]}_\text{reconstruction} - \beta\,\underbrace{\mathrm{KL}(q_\phi(\mathbf{z}|\mathbf{x})\,\|\,p(\mathbf{z}))}_\text{regularisation}$$
For a Gaussian prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0},\mathbf{I})$ the KL has a closed form:
$$\mathrm{KL} = -\frac{1}{2}\sum_{j=1}^d\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

In [ ]:
class VAEEncoder(nn.Module):
    """Outputs mu and log_var (not z directly)."""
    def __init__(self, latent_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 400), nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(400, latent_dim)
        self.fc_log_var = nn.Linear(400, latent_dim)

    def forward(self, x):
        h       = self.shared(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var


class VAE(nn.Module):
    def __init__(self, latent_dim=2, beta=1.0):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = Decoder(latent_dim)   # same decoder as AE
        self.beta    = beta

    def reparameterise(self, mu, log_var):
        """z = mu + sigma * eps,  eps ~ N(0, I)"""
        std = torch.exp(0.5 * log_var)  # sigma = exp(log_sigma) = exp(0.5 * log_var)
        eps = torch.randn_like(std)     # eps ~ N(0, I), same shape as std
        return mu + std * eps

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z           = self.reparameterise(mu, log_var)
        xhat        = self.decoder(z)
        return xhat, mu, log_var, z

    def elbo_loss(self, x, xhat, mu, log_var):
        # Reconstruction: binary cross-entropy (treats pixels as Bernoulli)
        recon = F.binary_cross_entropy(xhat, x, reduction='sum') / x.size(0)
        # KL divergence (closed form for Gaussian prior)
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
        return recon + self.beta * kl, recon, kl


def train_vae(model, loader, epochs=10, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    history = {'loss': [], 'recon': [], 'kl': []}
    for ep in range(1, epochs + 1):
        tot_loss = tot_recon = tot_kl = 0
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad()
            xhat, mu, log_var, _ = model(x)
            loss, recon, kl = model.elbo_loss(x, xhat, mu, log_var)
            loss.backward()
            opt.step()
            n = x.size(0)
            tot_loss  += loss.item()  * n
            tot_recon += recon.item() * n
            tot_kl    += kl.item()    * n
        N = len(loader.dataset)
        history['loss'].append(tot_loss / N)
        history['recon'].append(tot_recon / N)
        history['kl'].append(tot_kl / N)
        print(f'  Epoch {ep:2d}  loss={tot_loss/N:.2f}  recon={tot_recon/N:.2f}  KL={tot_kl/N:.2f}')
    return history


vae = VAE(LATENT_DIM, beta=1.0).to(DEVICE)
print('Training VAE ...')
vae_history = train_vae(vae, train_loader, epochs=10)

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ae_losses, label='AE (MSE)', color='steelblue')
axes[0].set_title('Plain AE — training loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(vae_history['loss'],  label='Total ELBO', color='darkorange')
axes[1].plot(vae_history['recon'], label='Reconstruction', color='crimson',     linestyle='--')
axes[1].plot(vae_history['kl'],    label='KL divergence',  color='mediumseagreen', linestyle=':')
axes[1].set_title('VAE — training losses')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Reconstruction quality

Compare original images against AE and VAE reconstructions.

In [ ]:
def get_reconstructions(model, loader, n=10, is_vae=True):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(DEVICE)
    with torch.no_grad():
        if is_vae:
            xhat, _, _, _ = model(x)
        else:
            xhat, _ = model(x)
    return x.cpu(), xhat.cpu()


def show_reconstructions(originals, ae_recons, vae_recons, n=10):
    fig, axes = plt.subplots(3, n, figsize=(n * 1.4, 4.5))
    labels = ['Original', 'AE', 'VAE']
    for i, (row_imgs, label) in enumerate(zip([originals, ae_recons, vae_recons], labels)):
        for j in range(n):
            axes[i, j].imshow(row_imgs[j].squeeze(), cmap='gray', vmin=0, vmax=1)
            axes[i, j].axis('off')
        axes[i, 0].set_ylabel(label, fontsize=12, rotation=90, labelpad=10)
    plt.suptitle('Reconstructions: Original vs AE vs VAE', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('reconstructions.png', dpi=120, bbox_inches='tight')
    plt.show()


originals, ae_recons  = get_reconstructions(ae,  test_loader, is_vae=False)
_,         vae_recons = get_reconstructions(vae, test_loader, is_vae=True)
show_reconstructions(originals, ae_recons, vae_recons)

## 6. Latent space visualisation — AE vs VAE

With a 2D latent space we can plot every test point.

**What to look for:**
- **AE**: fragmented clusters, large gaps between them. Sampling in the gaps produces garbage.
- **VAE**: smooth, continuous, roughly Gaussian blob. Classes overlap naturally. Sampling anywhere in the blob produces plausible digits.

In [ ]:
def encode_all(model, loader, is_vae=True):
    model.eval()
    zs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            if is_vae:
                mu, _ = model.encoder(x)
                zs.append(mu.cpu())
            else:
                zs.append(model.encoder(x).cpu())
            ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy()


ae_z,  ae_y  = encode_all(ae,  test_loader, is_vae=False)
vae_z, vae_y = encode_all(vae, test_loader, is_vae=True)

cmap = plt.cm.get_cmap('tab10', 10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, z, y, title in zip(axes, [ae_z, vae_z], [ae_y, vae_y], ['Plain AE', 'VAE']):
    for digit in range(10):
        mask = y == digit
        ax.scatter(z[mask, 0], z[mask, 1], s=4, alpha=0.5,
                   color=cmap(digit), label=str(digit))
    ax.set_title(f'{title} — latent space (test set)', fontsize=13)
    ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')
    ax.legend(markerscale=3, title='Digit', loc='upper right', fontsize=8)

# Draw the prior boundary on the VAE plot
theta = np.linspace(0, 2*np.pi, 200)
for r in [1, 2, 3]:
    axes[1].plot(r*np.cos(theta), r*np.sin(theta),
                 'k--', lw=0.8, alpha=0.3, label=f'{r}σ' if r==1 else '')
axes[1].annotate('1σ / 2σ / 3σ\nprior circles', xy=(0.98, 0.02),
                 xycoords='axes fraction', ha='right', va='bottom',
                 fontsize=7, color='gray')

plt.tight_layout()
plt.savefig('latent_space.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Generation — sampling from the prior

Sample $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decode.

For the AE, the same operation should produce mostly garbage because there is no prior — the encoder was never encouraged to fill the space around $\mathbf{0}$.

In [ ]:
def sample_from_prior(model, n=20, is_vae=True):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, LATENT_DIM).to(DEVICE)
        samples = model.decoder(z).cpu()
    return samples


def show_samples(ae_samples, vae_samples, n=20):
    fig, axes = plt.subplots(2, n, figsize=(n * 1.3, 3.5))
    for i, (samples, label) in enumerate(zip([ae_samples, vae_samples], ['AE', 'VAE'])):
        for j in range(n):
            axes[i, j].imshow(samples[j].squeeze(), cmap='gray', vmin=0, vmax=1)
            axes[i, j].axis('off')
        axes[i, 0].set_ylabel(label, fontsize=12, rotation=90, labelpad=10)
    plt.suptitle('Samples from z ~ N(0, I): AE (top) vs VAE (bottom)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('samples.png', dpi=120, bbox_inches='tight')
    plt.show()


ae_samples  = sample_from_prior(ae,  n=20)
vae_samples = sample_from_prior(vae, n=20)
show_samples(ae_samples, vae_samples)

## 8. Latent space interpolation

Encode two test images into $\mathbf{z}_1$ and $\mathbf{z}_2$, then decode along the straight line:
$$\mathbf{z}(\alpha) = (1-\alpha)\,\mathbf{z}_1 + \alpha\,\mathbf{z}_2, \qquad \alpha \in [0, 1]$$

In a VAE the interpolation should be smooth — every intermediate $\mathbf{z}$ is in a region
the decoder has seen during training. In a plain AE it may pass through an empty gap.

In [ ]:
def interpolate(model, x1, x2, steps=10, is_vae=True):
    model.eval()
    x1 = x1.unsqueeze(0).to(DEVICE)
    x2 = x2.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        if is_vae:
            z1, _ = model.encoder(x1)
            z2, _ = model.encoder(x2)
        else:
            z1 = model.encoder(x1)
            z2 = model.encoder(x2)
        alphas = torch.linspace(0, 1, steps).to(DEVICE)
        imgs = []
        for a in alphas:
            z = (1 - a) * z1 + a * z2
            imgs.append(model.decoder(z).cpu().squeeze())
    return imgs


def find_digit(dataset, digit):
    for x, y in dataset:
        if y == digit:
            return x

# Interpolate between a 3 and an 8
x_3 = find_digit(test_data, 3)
x_8 = find_digit(test_data, 8)
STEPS = 12

ae_interp  = interpolate(ae,  x_3, x_8, steps=STEPS, is_vae=False)
vae_interp = interpolate(vae, x_3, x_8, steps=STEPS, is_vae=True)

fig, axes = plt.subplots(2, STEPS, figsize=(STEPS * 1.3, 3.5))
for i, (interp, label) in enumerate(zip([ae_interp, vae_interp], ['AE', 'VAE'])):
    for j, img in enumerate(interp):
        axes[i, j].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(label, fontsize=12, rotation=90, labelpad=10)

axes[0, 0].set_title('3', fontsize=10)
axes[0, -1].set_title('8', fontsize=10)
plt.suptitle('Latent interpolation 3 → 8: AE (top) vs VAE (bottom)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('interpolation.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Latent grid — the digit manifold

Decode on a regular $20 \times 20$ grid of $\mathbf{z}$ values spanning $[-3, 3]^2$.
This gives a "map" of what the decoder has learned about each region of the latent space.

Each cell shows what the VAE would generate if we sampled exactly that $\mathbf{z}$.

In [ ]:
def latent_grid(model, grid_size=20, z_range=3.0):
    model.eval()
    lin = torch.linspace(-z_range, z_range, grid_size)
    z2, z1 = torch.meshgrid(lin, lin, indexing='ij')  # z1 = x-axis, z2 = y-axis
    z = torch.stack([z1.flatten(), z2.flatten()], dim=1).to(DEVICE)
    with torch.no_grad():
        imgs = model.decoder(z).cpu().squeeze().numpy()
    # Arrange into a grid image
    canvas = np.zeros((grid_size * 28, grid_size * 28))
    for idx, img in enumerate(imgs):
        r = idx // grid_size
        c = idx  % grid_size
        canvas[r*28:(r+1)*28, c*28:(c+1)*28] = img
    return canvas


grid = latent_grid(vae, grid_size=20)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid, cmap='gray', vmin=0, vmax=1, origin='upper')
ax.set_title('VAE latent grid — decoded $\mathbf{z}$ on $[-3,3]^2$ (20×20)', fontsize=13)

# Axis ticks in z-units
ticks = np.linspace(0, 20*28 - 28, 7)
labels = [f'{v:.1f}' for v in np.linspace(-3, 3, 7)]
ax.set_xticks(ticks); ax.set_xticklabels(labels)
ax.set_yticks(ticks); ax.set_yticklabels(labels[::-1])
ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')

plt.tight_layout()
plt.savefig('latent_grid.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. β-VAE — trading reconstruction for structure

The β-VAE (Higgins et al., 2017) multiplies the KL term by $\beta > 1$:
$$\mathcal{L}_\beta = \text{reconstruction} - \beta \cdot \text{KL}$$

- **β = 1**: standard VAE.
- **β > 1**: stronger regularisation → more Gaussian, more disentangled latent space, but worse reconstructions.
- **β < 1**: weaker regularisation → better reconstructions, messier latent space (closer to a plain AE).

We train three models with different β and compare their latent spaces.

In [ ]:
betas = [0.1, 1.0, 4.0]
beta_vaes = {}

for beta in betas:
    print(f'\nTraining β-VAE with β={beta} ...')
    m = VAE(LATENT_DIM, beta=beta).to(DEVICE)
    train_vae(m, train_loader, epochs=10)
    beta_vaes[beta] = m

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col, beta in enumerate(betas):
    model = beta_vaes[beta]

    # --- Latent space (top row) ---
    z_all, y_all = encode_all(model, test_loader, is_vae=True)
    ax = axes[0, col]
    for digit in range(10):
        mask = y_all == digit
        ax.scatter(z_all[mask, 0], z_all[mask, 1], s=3, alpha=0.4, color=cmap(digit))
    theta = np.linspace(0, 2*np.pi, 200)
    for r in [1, 2, 3]:
        ax.plot(r*np.cos(theta), r*np.sin(theta), 'k--', lw=0.6, alpha=0.3)
    ax.set_title(f'β = {beta}  —  latent space', fontsize=12)
    ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')

    # --- Latent grid (bottom row) ---
    grid = latent_grid(model, grid_size=16)
    axes[1, col].imshow(grid, cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'β = {beta}  —  latent grid', fontsize=12)
    axes[1, col].axis('off')

plt.suptitle('β-VAE comparison: latent spaces and grids', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('beta_vae_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Summary — what we learned

| Property | Plain AE | VAE | β-VAE (β > 1) |
|---|---|---|---|
| Latent space structure | Fragmented clusters | Smooth, Gaussian | More Gaussian, more disentangled |
| Sampling from prior | Garbage | Plausible digits | Plausible, more spread |
| Interpolation | May hit gaps | Smooth | Smooth |
| Reconstruction quality | Best | Good | Worse |
| Requires reparameterisation | No | Yes | Yes |
| KL term in loss | No | Yes (β=1) | Yes (β>1) |

**Key takeaways:**

1. **The reparameterisation trick** is what makes VAE training possible: it moves the randomness into $\boldsymbol{\varepsilon}$ (which does not depend on parameters), so gradients can flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$.

2. **The KL term** forces the encoder to produce distributions close to $\mathcal{N}(\mathbf{0},\mathbf{I})$. Without it, the model degenerates to a plain AE (it sets $\sigma \to 0$ to minimise reconstruction loss).

3. **The latent space is continuous** because nearby $\mathbf{z}$ values must decode to similar images — this is what makes interpolation smooth and generation meaningful.

4. **β controls the reconstruction–regularisation trade-off**: larger β imposes stronger structure but at the cost of reconstruction quality.

In [ ]:
# Exercise: try these experiments
# 1. Change LATENT_DIM to 10 and retrain. Can you still visualise the space? (Use PCA or t-SNE.)
# 2. Remove the KL term entirely (beta=0) — does the VAE become a plain AE?
# 3. Use a convolutional encoder/decoder. Does reconstruction quality improve?
# 4. Try the Celeb-A dataset instead of MNIST. Can you find a latent dimension
#    that encodes 'glasses' or 'smile'?